# Databricks Dynamic Ingestion - Step 00: Environment & Metadata Configuration
This notebook initializes the required databases (`bronze_db`, `silver_db`, `gold_db`, `rpt_db`) and creates the metadata driver table `bronze_db.ingestion_config` along with the audit table `bronze_db.ingestion_audit`.

### Target Architecture:
- **`bronze_db`**: Holds raw configurations, audit logs, and quarantine datasets.
- **`silver_db`**: Holds clean, schema-validated Delta tables.
- **`gold_db`**: Holds Star-Schema Fact & Dimension tables.
- **`rpt_db`**: Holds aggregated business reporting tables and analytical views.



In [0]:
# Create Medallion Architecture Databases
spark.sql("CREATE DATABASE IF NOT EXISTS bronze_db")
spark.sql("CREATE DATABASE IF NOT EXISTS silver_db")
spark.sql("CREATE DATABASE IF NOT EXISTS gold_db")
spark.sql("CREATE DATABASE IF NOT EXISTS rpt_db")

print("Databases created successfully: bronze_db, silver_db, gold_db, rpt_db")



In [0]:
# Create Audit Table in bronze_db
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_db.ingestion_audit (
    job_id STRING,
    target_table_db STRING,
    target_table_name STRING,
    source_location STRING,
    bronze_location STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    total_records_processed LONG,
    success_count LONG,
    failed_count LONG,
    status STRING,
    error_summary STRING
) USING DELTA
""")
print("Table bronze_db.ingestion_audit initialized successfully.")



In [0]:
# Populate Metadata Driver Table: bronze_db.ingestion_config
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from datetime import date

config_schema = StructType([
    StructField("data_entry_seq_no", IntegerType(), True),
    StructField("source_location", StringType(), True),
    StructField("bronze_location", StringType(), True),
    StructField("file_name_pattern", StringType(), True),
    StructField("target_format", StringType(), True),
    StructField("target_table_name", StringType(), True),
    StructField("target_table_db", StringType(), True),
    StructField("source_column_name", StringType(), True),
    StructField("target_column_name", StringType(), True),
    StructField("source_datatype", StringType(), True),
    StructField("target_datatype", StringType(), True),
    StructField("target_data_pattern", StringType(), True),
    StructField("col_seq_no", IntegerType(), True),
    StructField("data_entry_date", DateType(), True)
])

today = date.today()

config_data = [
    # 1. raw_customers -> silver_db.customers
    (1, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_customers.csv", "/tmp/bronze/raw_customers.csv", "raw_customers.csv", "delta", "customers", "silver_db", "id", "customer_id", "string", "string", "NOT_NULL", 1, today),
    (2, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_customers.csv", "/tmp/bronze/raw_customers.csv", "raw_customers.csv", "delta", "customers", "silver_db", "name", "customer_name", "string", "string", None, 2, today),

    # 2. raw_orders -> silver_db.orders
    (3, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_orders.csv", "/tmp/bronze/raw_orders.csv", "raw_orders.csv", "delta", "orders", "silver_db", "id", "order_id", "string", "string", "NOT_NULL", 1, today),
    (4, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_orders.csv", "/tmp/bronze/raw_orders.csv", "raw_orders.csv", "delta", "orders", "silver_db", "customer", "customer_id", "string", "string", "NOT_NULL", 2, today),
    (5, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_orders.csv", "/tmp/bronze/raw_orders.csv", "raw_orders.csv", "delta", "orders", "silver_db", "ordered_at", "ordered_at", "string", "timestamp", "yyyy-MM-ddTHH:mm:ss", 3, today),
    (6, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_orders.csv", "/tmp/bronze/raw_orders.csv", "raw_orders.csv", "delta", "orders", "silver_db", "store_id", "store_id", "string", "string", "NOT_NULL", 4, today),
    (7, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_orders.csv", "/tmp/bronze/raw_orders.csv", "raw_orders.csv", "delta", "orders", "silver_db", "subtotal", "subtotal_cents", "string", "integer", ">=0", 5, today),
    (8, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_orders.csv", "/tmp/bronze/raw_orders.csv", "raw_orders.csv", "delta", "orders", "silver_db", "tax_paid", "tax_paid_cents", "string", "integer", ">=0", 6, today),
    (9, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_orders.csv", "/tmp/bronze/raw_orders.csv", "raw_orders.csv", "delta", "orders", "silver_db", "order_total", "order_total_cents", "string", "integer", ">=0", 7, today),

    # 3. raw_items -> silver_db.order_items
    (10, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_items.csv", "/tmp/bronze/raw_items.csv", "raw_items.csv", "delta", "order_items", "silver_db", "id", "item_id", "string", "string", "NOT_NULL", 1, today),
    (11, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_items.csv", "/tmp/bronze/raw_items.csv", "raw_items.csv", "delta", "order_items", "silver_db", "order_id", "order_id", "string", "string", "NOT_NULL", 2, today),
    (12, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_items.csv", "/tmp/bronze/raw_items.csv", "raw_items.csv", "delta", "order_items", "silver_db", "sku", "sku", "string", "string", "NOT_NULL", 3, today),

    # 4. raw_products -> silver_db.products
    (13, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_products.csv", "/tmp/bronze/raw_products.csv", "raw_products.csv", "delta", "products", "silver_db", "sku", "sku", "string", "string", "NOT_NULL", 1, today),
    (14, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_products.csv", "/tmp/bronze/raw_products.csv", "raw_products.csv", "delta", "products", "silver_db", "name", "product_name", "string", "string", None, 2, today),
    (15, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_products.csv", "/tmp/bronze/raw_products.csv", "raw_products.csv", "delta", "products", "silver_db", "type", "product_type", "string", "string", None, 3, today),
    (16, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_products.csv", "/tmp/bronze/raw_products.csv", "raw_products.csv", "delta", "products", "silver_db", "price", "price_cents", "string", "integer", ">=0", 4, today),
    (17, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_products.csv", "/tmp/bronze/raw_products.csv", "raw_products.csv", "delta", "products", "silver_db", "description", "description", "string", "string", None, 5, today),

    # 5. raw_stores -> silver_db.stores
    (18, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_stores.csv", "/tmp/bronze/raw_stores.csv", "raw_stores.csv", "delta", "stores", "silver_db", "id", "store_id", "string", "string", "NOT_NULL", 1, today),
    (19, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_stores.csv", "/tmp/bronze/raw_stores.csv", "raw_stores.csv", "delta", "stores", "silver_db", "name", "store_name", "string", "string", None, 2, today),
    (20, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_stores.csv", "/tmp/bronze/raw_stores.csv", "raw_stores.csv", "delta", "stores", "silver_db", "opened_at", "opened_at", "string", "timestamp", "yyyy-MM-ddTHH:mm:ss", 3, today),
    (21, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_stores.csv", "/tmp/bronze/raw_stores.csv", "raw_stores.csv", "delta", "stores", "silver_db", "tax_rate", "tax_rate", "string", "float", ">=0", 4, today),

    # 6. raw_supplies -> silver_db.supplies
    (22, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_supplies.csv", "/tmp/bronze/raw_supplies.csv", "raw_supplies.csv", "delta", "supplies", "silver_db", "id", "supply_id", "string", "string", "NOT_NULL", 1, today),
    (23, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_supplies.csv", "/tmp/bronze/raw_supplies.csv", "raw_supplies.csv", "delta", "supplies", "silver_db", "name", "supply_name", "string", "string", None, 2, today),
    (24, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_supplies.csv", "/tmp/bronze/raw_supplies.csv", "raw_supplies.csv", "delta", "supplies", "silver_db", "cost", "cost_cents", "string", "integer", ">=0", 3, today),
    (25, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_supplies.csv", "/tmp/bronze/raw_supplies.csv", "raw_supplies.csv", "delta", "supplies", "silver_db", "perishable", "is_perishable", "string", "boolean", None, 4, today),
    (26, "https://raw.githubusercontent.com/dbt-labs/jaffle-shop/refs/heads/main/seeds/jaffle-data/raw_supplies.csv", "/tmp/bronze/raw_supplies.csv", "raw_supplies.csv", "delta", "supplies", "silver_db", "sku", "product_sku", "string", "string", None, 5, today),
]

df_config = spark.createDataFrame(config_data, config_schema)
df_config.write.format("delta").mode("overwrite").saveAsTable("bronze_db.ingestion_config")

print("Table bronze_db.ingestion_config successfully populated.")



In [0]:
# Verify Ingestion Config Contents
display(spark.sql("SELECT target_table_name, COUNT(*) as col_count FROM bronze_db.ingestion_config GROUP BY target_table_name ORDER BY target_table_name"))

